# pymadagascar Learning Guide

This notebook is a learning and usage entry point for pymadagascar. It is intentionally lightweight: mostly Markdown, a few optional code snippets, no saved outputs, and no dependency on external data, WSL, original Madagascar, GPU, MPI, CUDA, PETSc, or the optional C++ extension.

The authoritative project documents remain the eight Markdown files in docs. This notebook is not an API authority, not a coverage authority, and not a release-blocking executed notebook. It is a readable guide that Codex may update when new user-visible capabilities, algorithm topics, important workflows, or formulas materially improve the learning path.

## 1. Project positioning

pymadagascar is a Python-friendly local RSF/geophysics toolkit. It is not a complete clone of the upstream Madagascar project and does not try to migrate thousands of commands at once. The practical goal is a small, durable package for local RSF data handling, geophysical processing experiments, examples, tests, and documentation.

Project principles to keep in mind:

- pure Python must always remain usable;
- C++/hybrid code is optional acceleration, never a hard dependency;
- stable/root API changes are deliberately conservative;
- original Madagascar comparisons are optional compatibility checks;
- large imaging, MPI/CUDA/PETSc, IWAVE/RVL, VPlot, SCons/book, and full user command migration are not near-term targets.

## 2. Package structure

High-level areas:

- pymadagascar.io and pymadagascar.core: RSF headers, sidecars, axes, hypercubes, and metadata.
- pymadagascar.generic: generic array transforms, statistics, sampling, headers, operators, least-squares helpers, and prototype solvers.
- pymadagascar.signal: FFT, filtering, convolution/correlation, conditioning, spectral QC, FIR, and related signal tools.
- pymadagascar.seismic: seismic gather utilities such as gain, mute/mutter, stack, NMO, semblance, FK, and Radon/slant-stack prototypes.
- pymadagascar.cli: selected Madagascar-style command entry points and module-only command surfaces.
- pymadagascar.testing: internal fixtures, metrics, geometry helpers, and optional original-Madagascar comparison infrastructure.
- examples: focused demos plus longer local workflows under examples/my_workflows.
- docs: the authoritative Markdown docs plus this learning notebook.

Use the Markdown docs for current project authority: status, compatibility, coverage, testing, limitations, changelog, user guide, and documentation README. Use this notebook as a softer map for learning.

## 3. Quick start

Install the package in editable pure-Python mode from the repository root:

    python -m pip install -e ".[test]"

Run tests when appropriate for the risk of the change:

    python -m pytest -q
    python tools/check_release.py
    python tools/check_cli_inventory.py
    python tools/check_docs_commands.py
    python tools/check_examples_inventory.py
    python tools/check_learning_notebook.py

Run a CLI command from an installed environment, for example help text or a focused transform:

    pymada-spike --help
    pymada-info --help

Prefer project docs for exact command contracts and examples because command surfaces are intentionally selective.

## 4. Python API basics

The stable root API is intentionally small. For many workflows you start with RSF I/O and RSFData, then use explicit module APIs for prototype or specialized behavior.

Minimal RSFData sketch:

    import numpy as np
    from pymadagascar import RSFData
    data = np.arange(12, dtype=np.float32).reshape(3, 4)
    rsf = RSFData(data)
    scaled = rsf * 2.0

For file-backed work, use read and write from the root package or the lower-level RSF I/O functions. RSF headers preserve axis metadata through n#, o#, d#, labels, units, dtype, and sidecar layout. The ascii_float subset is useful for small readable fixtures and tests; binary RSF sidecars are the ordinary data path.

In [ ]:
import numpy as np
from pymadagascar import RSFData

data = np.arange(6, dtype=np.float32).reshape(2, 3)
rsf = RSFData(data)
rsf.shape, rsf.dtype

## 5. Topic index for geophysical processing

Useful learning route by topic:

- RSF I/O: headers, sidecars, axis metadata, RSFData, read, write, ascii_float, and binary data.
- Basic CLI transforms: spike, window, math, dd, cat, transp, get, info, attr, and related generic tools.
- Signal and QC: FFT, filters, tapers, spectra, envelope, convolution/correlation, local RMS, PSD/CSD, coherence, spectrogram, SNR, Welch estimates, FIR filters, and band energy.
- DAS workflow: retained local workflow prototype combining RSF, signal, FK-style filtering, plotting, and simple picking output.
- Seismic workflows: trace/panel/gather contracts, small gather geometry, NMO, semblance, FK, Radon/slant stack, and integrated regression workflows.
- Inversion / operator foundation: LinearOperator, dot tests, composition, LeastSquaresProblem, regularization, preconditioner contracts, CG/CGNR/CGLS, and right/model-space preconditioning.

## 6. Inversion / operator foundation

The current operator foundation is a small in-memory prototype layer, not a production inversion framework. Import directly from pymadagascar.generic modules or the compatibility re-export layer pymadagascar.generic.linear_operator.

Core ideas:

- LinearOperator: abstract forward/Hermitian-adjoint interface.
- MatrixOperator: wraps a dense matrix for small tests and demos.
- Operator composition and stacking: build small systems such as augmented least-squares operators.
- Dot tests: check adjoint consistency, including complex Hermitian behavior.
- LeastSquaresProblem: stores the data operator, right-hand side, optional regularization, and diagnostics.
- SolverHistory and SolverResult: record bounded prototype solver diagnostics without expanding the stable root API.

## 7. Regularization, CGLS, and preconditioners

Unregularized least squares solves:

    min_x 0.5 * ||A x - b||^2

Regularized least squares is represented by an augmented system:

    min_x 0.5 * ||A x - b||^2 + 0.5 * lambda^2 * ||L x||^2

equivalently:

    [A; lambda L] x ~= [b; 0]

The prototype CGLS helpers work on small in-memory systems. Regularization changes the objective. A preconditioner changes the variable scaling and convergence behavior while preserving the model-space objective diagnostics.

## 8. Right/model-space preconditioning

The right-preconditioned CGLS prototype uses a model-space map M from latent variables z to model variables x:

    x = M z

Without regularization, the recurrence runs on:

    min_z 0.5 * ||A M z - b||^2

With regularization, the augmented system becomes:

    [A M; lambda L M] z ~= [b; 0]

The final result is returned in model space as x = M z, not latent space. Diagnostics still evaluate data residuals, regularization residuals, and objective terms using the model-space x.

In [ ]:
import numpy as np
from pymadagascar.generic.preconditioners import DiagonalPreconditioner
from pymadagascar.generic.solvers import run_cgls

A = np.array([[1.0, 0.0], [0.0, 0.25]])
x_true = np.array([2.0, -4.0])
b = A @ x_true
M = DiagonalPreconditioner([1.0, 4.0])
result = run_cgls(A, b, maxiter=4, preconditioner=M)
result.final_model

## 9. CGLS and LSQR for small least-squares problems

The small inversion helpers are meant for learning and deterministic local experiments before any production-scale inversion design. Their shared least-squares target is

```text
min_x 0.5 * ||A x - b||^2
```

Regularization is represented through the same augmented system used by the solver tests:

```text
min_x 0.5 * ||[A; lambda L] x - [b; 0]||^2
```

Current CGLS status:

- `run_cgls` and `run_cgls_problem` are bounded direct-module prototypes.
- Right/model-space preconditioning is supported for CGLS with `x = M z`.
- Results and objective diagnostics remain in model space; convergence metadata says when a residual lives in latent space.

Current LSQR status:

- `run_lsqr` and `run_lsqr_problem` are bounded pure-Python direct-module prototypes.
- LSQR supports unpreconditioned small least-squares problems and regularized `LeastSquaresProblem` objects.
- Nonzero model-space `x0` uses a shifted residual correction:

```text
r0 = b_aug - A_aug x0
solve A_aug dx ~= r0
final x = x0 + dx
```

- Preconditioned LSQR is intentionally not enabled yet.
- LSQR has no CLI and is not part of the root/stable API.

A practical rule of thumb: use CGLS when studying normal equations, gradient convergence, or the current right-preconditioner contract; use LSQR when studying Golub-Kahan bidiagonalization-style least-squares behavior on tiny deterministic systems.


## 10. Learning route

A practical route for future study:

1. Learn RSF data first: headers, sidecars, axis conventions, dtype, and RSFData.
2. Learn the basic CLI and Python API surface: what is stable, what is module-only, and what is prototype.
3. Work through signal and seismic workflows: small deterministic fixtures before field-scale assumptions.
4. Study operator / inversion foundations: dot tests, LinearOperator, LeastSquaresProblem, regularization, CGLS, and preconditioner semantics.
5. Treat optional hybrid acceleration and original Madagascar comparisons as validation/support layers, not as pure-Python requirements.

When a future change adds a user-visible capability, algorithm topic, important workflow, or useful formula explanation, consider updating this notebook. For tiny internal changes, avoid documentation noise.